# 01 · Dataset download & inventory
Downloads the public *Degraded Artefact Image Data Set* (Stoean et al., **CC BY 4.0**, ~700 MB) and inventories its folder layout so every later stage is written against what is **really** inside — not assumptions.

Resumable: if the session dies mid-download, just re-run.

In [ ]:
# --- Bootstrap: mount Drive, get the code, make `heritage` importable ---
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/artefact-restoration-3d.git"  # <- set after you push to GitHub
REPO_DIR = Path("/content/artefact-restoration-3d")
DRIVE_CODE = Path("/content/drive/MyDrive/artefact-restoration-3d/code")   # fallback: upload the repo folder here

from google.colab import drive
drive.mount("/content/drive")

if not REPO_DIR.exists():
    if "YOUR_USERNAME" not in REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif DRIVE_CODE.exists():
        shutil.copytree(DRIVE_CODE, REPO_DIR)
    else:
        raise SystemExit("Set REPO_URL (after pushing to GitHub) or upload the repo folder to " + str(DRIVE_CODE))
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)
print("Repo ready at", REPO_DIR)


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!python scripts/download_dataset.py

In [ ]:
from heritage import paths, inventory
import json
cfg = paths.load_config(); P = paths.get_paths(cfg).ensure()
records = inventory.inventory(P.data, max_depth=8)
from IPython.display import Markdown, display
display(Markdown(inventory.format_markdown(records)))
P.reports.mkdir(parents=True, exist_ok=True)
(P.reports / 'dataset_inventory.json').write_text(json.dumps(records, indent=2))
(P.reports / 'dataset_inventory.md').write_text(inventory.format_markdown(records))
print('Saved inventory to', P.reports)

### Look at the data
One image from every folder that holds images, so we can see what each folder is (raw frames, masks, ground truth, pre-computed inpaintings ...).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from heritage.ids import list_images
folders = [r['folder'] for r in records if r['n_images'] > 0]
print(len(folders), 'image folders')
cols = 4; rows = (len(folders) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3 * rows), squeeze=False)
for ax in axes.ravel(): ax.axis('off')
for ax, f in zip(axes.ravel(), folders):
    imgs = list_images(P.data / f)
    im = Image.open(imgs[len(imgs) // 2]).convert('RGB'); im.thumbnail((480, 300))
    ax.imshow(im); ax.set_title(f[-40:] + f'  [{len(imgs)}]', fontsize=7)
plt.tight_layout()
(P.figures).mkdir(parents=True, exist_ok=True)
plt.savefig(P.figures / 'dataset_overview.png', dpi=130); plt.show()

### Next
Download `dataset_inventory.md` from the `reports` folder on your Drive (or copy the table above) and send it to me. I use it to finalise the exact folder paths for notebook 02 (LaMa inpainting) — particularly whether the archive already contains `gt_inpainted/` (LaMa / Stable Diffusion results for the 29 ground-truth frames) or only raw frames and masks, which decides what we regenerate.